# 03 — Position-Specific Drill Analysis

**Purpose**: Stratify Combine tracking features by position group.
Identify which positions show the most signal in which drills.
Begin building position-specific candidate hypotheses.

**Key question per position group**:
- WR/TE: Do route-like directional changes during the Combine correlate with NFL separation?
- Edge/DL: Does first-step burst predict NFL get-off and pressure rate?
- OL: Does lateral agility / mirror movement predict pass-protection efficiency?
- CB/S: Does acceleration/deceleration pattern predict coverage ability?

> At this stage we generate hypotheses, not final conclusions.

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ensure_dirs, FIGURES_DIR, COMBINE_FEATURES_PARQUET, PLAYER_MASTER_PARQUET, DATA_PROCESSED
from src.data_loading import load_players, load_combine_results, load_combine_tracking
from src.player_mapping import assign_position_groups, build_master_player_table
from src.combine_features import aggregate_player_features
from src.statistics import spearman_with_ci, compare_groups, mannwhitney_matrix, build_correlation_summary
from src.visualization import plot_combine_vs_nfl, plot_correlation_heatmap
from src.config import MIN_SAMPLE_SIZE

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')
ensure_dirs()
print('Setup complete.')

## 1. Load Pre-computed Features

In [ ]:
# Load from Parquet (generated in notebook 02)
if COMBINE_FEATURES_PARQUET.exists():
    features_df = pd.read_parquet(COMBINE_FEATURES_PARQUET)
    print(f'Loaded combine features: {features_df.shape}')
else:
    print('WARNING: combine_features.parquet not found — running feature extraction now...')
    from src.combine_features import extract_features_from_tracking
    tracking = load_combine_tracking()
    features_df = extract_features_from_tracking(tracking)
    features_df.to_parquet(COMBINE_FEATURES_PARQUET, index=False)

players = load_players()
players = assign_position_groups(players)
combine_results = load_combine_results(cols=None)

# Build master table
master = build_master_player_table(players, combine_results=combine_results)
master.to_parquet(PLAYER_MASTER_PARQUET, index=False)
print(f'Master player table: {master.shape}')

# Aggregate features to player level
player_features = aggregate_player_features(features_df, agg_strategy='best_attempt')

# Add position group
pos_map = players[['nfl_id', 'position', 'position_group']].drop_duplicates('nfl_id')
player_features = player_features.merge(pos_map, on='nfl_id', how='left')

print(f'Player-level features: {player_features.shape}')
print(f'Position groups present: {player_features["position_group"].value_counts().to_dict() if "position_group" in player_features.columns else "N/A"}')

## 2. Position Group Sample Sizes

Before any analysis, confirm sample sizes. We need >= 10 players per group to report findings.

In [ ]:
if 'position_group' in player_features.columns:
    size_table = player_features.groupby('position_group').agg(
        n_players=('nfl_id', 'nunique'),
        n_attempts=('nfl_id', 'count'),
    ).sort_values('n_players', ascending=False)

    print('=== Position Group Sample Sizes ===')
    print(f'Minimum required for reporting: {MIN_SAMPLE_SIZE}')
    display(size_table)

    viable_groups = size_table[size_table['n_players'] >= MIN_SAMPLE_SIZE].index.tolist()
    print(f'\nViable position groups (n >= {MIN_SAMPLE_SIZE}): {viable_groups}')
else:
    viable_groups = []
    print('position_group column not found.')

## 3. WR / TE Analysis

**Hypothesis being tested**: WRs with higher direction-change rates and re-acceleration ability
during Combine drills will show better NFL separation and YAC.

In [ ]:
# Filter to WR/TE
wr_te_feats = player_features[
    player_features['position_group'].isin(['WR', 'TE'])
].copy() if 'position_group' in player_features.columns else pd.DataFrame()

if len(wr_te_feats) >= MIN_SAMPLE_SIZE:
    print(f'WR/TE analysis population: {len(wr_te_feats)} player-drill rows')
    print(f'Unique players: {wr_te_feats["nfl_id"].nunique()}')

    # Key features for WR/TE
    wr_te_key_feats = [
        c for c in [
            'max_speed', 'total_direction_change', 'direction_change_rate',
            'best_peak_reaccel', 'best_brake_magnitude', 'best_transition_time',
            'movement_efficiency', 'mean_speed', 'n_cut_sequences',
            'first_step_acceleration', 'time_to_75pct_peak',
        ] if c in wr_te_feats.columns
    ]

    print(f'\nKey features available: {wr_te_key_feats}')

    # Summary statistics
    display(wr_te_feats[wr_te_key_feats].describe())

    # Compare WR vs TE on these features
    if 'position' in wr_te_feats.columns:
        for feat in wr_te_key_feats[:5]:
            comparison = compare_groups(wr_te_feats, 'position', feat, min_n_per_group=5)
            if not comparison.empty:
                print(f'\n{feat}:')
                display(comparison)
else:
    print(f'Insufficient WR/TE data (n={len(wr_te_feats)}, need {MIN_SAMPLE_SIZE})')

## 4. Edge / DL Analysis

**Hypothesis being tested**: Edge rushers with higher first-step acceleration and burst impulse
at the Combine will generate faster get-off and more quick pressure in the NFL.

In [ ]:
edge_feats = player_features[
    player_features['position_group'].isin(['Edge', 'DT'])
].copy() if 'position_group' in player_features.columns else pd.DataFrame()

if len(edge_feats) >= MIN_SAMPLE_SIZE:
    print(f'Edge/DT analysis population: {len(edge_feats)} player-drill rows')
    print(f'Unique players: {edge_feats["nfl_id"].nunique()}')

    edge_key_feats = [
        c for c in [
            'first_step_acceleration', 'burst_impulse_0_5s', 'burst_impulse_0_1s',
            'time_to_50pct_peak', 'time_to_75pct_peak', 'max_acceleration',
            'max_speed', 'mean_acceleration', 'movement_efficiency',
        ] if c in edge_feats.columns
    ]

    print(f'Key features: {edge_key_feats}')
    display(edge_feats[edge_key_feats].describe())

    # Compare Edge vs DT
    if 'position_group' in edge_feats.columns:
        for feat in edge_key_feats[:4]:
            comparison = compare_groups(edge_feats, 'position_group', feat, min_n_per_group=5)
            if not comparison.empty:
                print(f'\n{feat}:')
                display(comparison)
else:
    print(f'Insufficient Edge/DT data (n={len(edge_feats)}, need {MIN_SAMPLE_SIZE})')

## 5. OL Analysis

**Hypothesis being tested**: OL with better lateral agility (low efficiency in lateral drills,
high direction change rate) will allow fewer pressures.

In [ ]:
ol_feats = player_features[
    player_features['position_group'] == 'OL'
].copy() if 'position_group' in player_features.columns else pd.DataFrame()

if len(ol_feats) >= MIN_SAMPLE_SIZE:
    print(f'OL analysis population: {len(ol_feats)} player-drill rows')
    print(f'Unique players: {ol_feats["nfl_id"].nunique()}')

    ol_key_feats = [
        c for c in [
            'total_direction_change', 'direction_change_rate', 'movement_efficiency',
            'best_brake_magnitude', 'best_transition_time', 'mean_deceleration',
            'max_deceleration', 'first_step_acceleration',
        ] if c in ol_feats.columns
    ]

    display(ol_feats[ol_key_feats].describe())
else:
    print(f'Insufficient OL data (n={len(ol_feats)}, need {MIN_SAMPLE_SIZE})')

## 6. CB / Safety Analysis

In [ ]:
db_feats = player_features[
    player_features['position_group'].isin(['CB', 'S', 'DB'])
].copy() if 'position_group' in player_features.columns else pd.DataFrame()

if len(db_feats) >= MIN_SAMPLE_SIZE:
    print(f'CB/S analysis population: {len(db_feats)} rows | {db_feats["nfl_id"].nunique()} players')

    db_key_feats = [
        c for c in [
            'max_speed', 'total_direction_change', 'best_peak_reaccel',
            'first_step_acceleration', 'direction_change_rate', 'movement_efficiency',
        ] if c in db_feats.columns
    ]

    display(db_feats[db_key_feats].describe())
else:
    print(f'Insufficient CB/S data (n={len(db_feats)})')

## 7. Cross-Position Feature Comparison

**Research question**: Are certain features universally important (across positions), or highly position-specific?

In [ ]:
universal_feats = [
    c for c in ['max_speed', 'first_step_acceleration', 'total_direction_change', 'movement_efficiency']
    if c in player_features.columns
]

if 'position_group' in player_features.columns and universal_feats:
    for feat in universal_feats:
        valid = player_features[['position_group', feat]].dropna()
        if len(valid) < 10:
            continue

        order = valid.groupby('position_group')[feat].median().sort_values(ascending=False).index

        fig, ax = plt.subplots(figsize=(12, 5))
        sns.boxplot(data=valid, x='position_group', y=feat, order=order, palette='Blues_d', ax=ax)
        ax.set_title(f'{feat} Across Position Groups', fontweight='bold')
        ax.set_xlabel('Position Group')
        ax.set_ylabel(feat)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f'position_comparison_{feat}.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('Insufficient data for cross-position comparison.')

## 8. Drill-Position Heatmap

Which drills have which position groups? Are some drills exclusive to certain positions?

In [ ]:
if 'drill_type' in features_df.columns and 'position_group' in player_features.columns:
    # Merge drill info with position info
    drill_pos = features_df[['nfl_id', 'drill_type']].merge(
        pos_map, on='nfl_id', how='left'
    )
    if 'position_group' in drill_pos.columns:
        pivot = (
            drill_pos.groupby(['position_group', 'drill_type'])['nfl_id']
            .nunique()
            .unstack(fill_value=0)
        )

        fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 1.5), max(5, len(pivot) * 0.5)))
        sns.heatmap(
            pivot, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Number of players'},
        )
        ax.set_title('Players per Position Group × Drill Type', fontweight='bold')
        ax.set_xlabel('Drill Type')
        ax.set_ylabel('Position Group')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'position_drill_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()

        display(pivot)
else:
    print('Cannot build drill-position heatmap: missing columns.')

## 9. Feature Correlation Within Position Groups

Within each position group, which tracking features are most correlated with each other?
Highly correlated features may be redundant.

In [ ]:
from src.statistics import correlation_matrix
from src.visualization import plot_correlation_heatmap

key_feats = [
    c for c in [
        'max_speed', 'first_step_acceleration', 'total_direction_change',
        'movement_efficiency', 'best_peak_reaccel', 'time_to_75pct_peak',
        'direction_change_rate', 'n_cut_sequences', 'mean_deceleration',
    ] if c in player_features.columns
]

if len(key_feats) >= 3 and 'position_group' in player_features.columns:
    for group in viable_groups[:3]:  # limit to top 3 viable groups
        group_data = player_features[player_features['position_group'] == group]
        n = len(group_data)
        if n < MIN_SAMPLE_SIZE:
            continue

        # Intra-feature correlation matrix
        avail_feats = [f for f in key_feats if f in group_data.columns]
        corr_mat = correlation_matrix(group_data, x_cols=avail_feats, y_cols=avail_feats)

        if not corr_mat.empty:
            plot_correlation_heatmap(
                corr_mat,
                title=f'{group} — Feature Intercorrelation (n={n})',
                filename=f'feature_intercorr_{group}.png',
            )
            plt.show()
else:
    print('Insufficient features or position groups for correlation analysis.')

## 10. Summary: Most Promising Position × Drill × Feature Combinations

Based on the analysis above, summarize what looks promising.

In [ ]:
print("""
=== POSITION × DRILL × FEATURE SUMMARY ===

This is based on the EDA above. Fill in observations after running the notebook.

Each row below should be completed with ACTUAL numbers from the cells above.
Do not invent values.
""")

# Print sample sizes for each position × drill combination
if 'drill_type' in features_df.columns and 'position_group' in player_features.columns:
    drill_pos_summary = (
        features_df[['nfl_id', 'drill_type']]
        .merge(pos_map, on='nfl_id', how='left')
        .groupby(['position_group', 'drill_type'])['nfl_id']
        .nunique()
        .reset_index(name='n_players')
        .query(f'n_players >= {MIN_SAMPLE_SIZE}')
        .sort_values(['position_group', 'n_players'], ascending=[True, False])
    )
    print('Position × Drill combinations with sufficient sample size:')
    display(drill_pos_summary)
else:
    print('NOT YET COMPUTED — run cells above first.')